# Ocular Sentinel - Cloud VLM Node (AMD ROCm)

This notebook is designed for a single-click run to preserve your 8-hour quota. 
It will install the necessary tunnel software, boot up the **Qwen2-VL-7B-Instruct** model on the AMD GPU using vLLM, and expose the API endpoint securely to your local PC.

**Instructions:**
1. Click **Cell -> Run All**.
2. Scroll to the bottom of the second cell to get your `ngrok` URL.
3. Wait for the model to finish loading (you will see `Uvicorn running on http://0.0.0.0:8000` in the logs).
4. Copy the URL into your local PC's terminal as the `VLLM_API_URL` environment variable.

In [ ]:
%pip install pyngrok qwen-vl-utils[decord]

In [ ]:
import subprocess
import time
import sys
import os
from pyngrok import ngrok

# Disable NCCL P2P which can cause silent hangs on single AMD GPUs during initialization
os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

# ==================================================================
# ⚠️ NGROK REQUIRES A FREE AUTH TOKEN TO CREATE TUNNELS
# 1. Go to: https://dashboard.ngrok.com/get-started/your-authtoken
# 2. Log in with Google/GitHub (takes 10 seconds)
# 3. Paste the token below:
# ==================================================================
NGROK_TOKEN = "PASTE_YOUR_TOKEN_HERE"

if NGROK_TOKEN == "PASTE_YOUR_TOKEN_HERE":
    raise ValueError("❌ ERROR: You must paste your free Ngrok Token on line 17 before running this cell!")

ngrok.set_auth_token(NGROK_TOKEN)

print("Starting vLLM server on port 8000 (Qwen2-VL-7B-Instruct)...")
print("This will take a few minutes to download the model weights to the AMD GPUs.")

# Start vLLM in the background and pipe the output so we can stream it
# We use sys.executable to ensure we use the exact Python environment that Jupyter is running in
# --enforce-eager prevents ROCm Graph Compilation hangs on Notebooks
vllm_process = subprocess.Popen(
    [sys.executable, "-m", "vllm.entrypoints.openai.api_server", 
     "--model", "Qwen/Qwen2-VL-7B-Instruct", 
     "--dtype", "bfloat16", 
     "--port", "8000",
     "--max-model-len", "4096",
     "--limit-mm-per-prompt", '{"video": 1}',
     "--enforce-eager",
     "--trust-remote-code"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=os.environ.copy()
)

# Give it 10 seconds to initialize the server socket before opening the tunnel
time.sleep(10)

print("\nOpening ngrok tunnel...")
public_url = ngrok.connect(8000)
api_url = f"{public_url.public_url}/v1"

print("====================================================================")
print("SUCCESS! TUNNEL IS OPEN AND SERVER IS BOOTING.")
print(f"YOUR AMD VLM ENDPOINT URL: {api_url}")
print("\nOn your local PC, run:")
print(f'$env:VLLM_API_URL="{api_url}"; python main.py')
print("====================================================================\n")

print("Streaming vLLM server logs (wait until you see 'Application startup complete'):")
for line in vllm_process.stdout:
    print(line, end="")
